Sales Man Problem

In [3]:
import itertools

# Cities: 0 = City A (Hub), 1 = City B, 2 = City C, 3 = City D
cities = ["City A", "City B", "City C", "City D"]

# Travel time matrix (in minutes) from the delivery van problem
dist = [
    [0,  15, 25, 30],  # From City A
    [15, 0,  20, 35],  # From City B
    [25, 20, 0,  10],  # From City C
    [30, 35, 10, 0],   # From City D
]

def solve_tsp_held_karp(distance_matrix, start=0):
    """
    Solves the Travelling Salesman Problem starting and ending at `start`
    using Held-Karp dynamic programming.
    Returns (min_cost, path_indices).
    """
    n = len(distance_matrix)
    all_visited = (1 << n) - 1

    # dp[mask][u] = minimum cost to reach subset `mask` ending at city `u`
    INF = float("inf")
    dp = [[INF] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]

    dp[1 << start][start] = 0

    for mask in range(1 << n):
        if not (mask & (1 << start)):
            continue
        for u in range(n):
            if not (mask & (1 << u)) or dp[mask][u] == INF:
                continue
            for v in range(n):
                if mask & (1 << v):
                    continue
                next_mask = mask | (1 << v)
                new_cost = dp[mask][u] + distance_matrix[u][v]
                if new_cost < dp[next_mask][v]:
                    dp[next_mask][v] = new_cost
                    parent[next_mask][v] = u

    # Close the tour back to `start`
    min_cost = INF
    last_city = -1
    for u in range(n):
        if u == start or dp[all_visited][u] == INF:
            continue
        return_cost = dp[all_visited][u] + distance_matrix[u][start]
        if return_cost < min_cost:
            min_cost = return_cost
            last_city = u

    # Reconstruct optimal tour
    path = []
    curr_mask = all_visited
    curr_city = last_city
    while curr_city != -1:
        path.append(curr_city)
        prev_city = parent[curr_mask][curr_city]
        curr_mask ^= (1 << curr_city)
        curr_city = prev_city

    path.reverse()
    path.append(start)  # Return to starting hub
    return min_cost, path

min_time, route_indices = solve_tsp_held_karp(dist, start=0)
route_names = [cities[i] for i in route_indices]

print(f"Minimum Travel Time: {min_time} minutes")
print("Optimal Route:      " + " -> ".join(route_names))

Minimum Travel Time: 75 minutes
Optimal Route:      City A -> City D -> City C -> City B -> City A


Read Me

A Google Colab Python implementation solving the 4-city delivery van Travelling Salesman Problem using Held-Karp dynamic programming with bitmasks. It computes the minimum round-trip travel time (75 minutes) and reconstructs the optimal route in $O(n^2 \cdot 2^n)$ time complexity.